# Burnout Risk Multiclass Classification

**Dataset:** Impact of AI on Students  
**Target:** `Burnout_Risk_Level`  
**Task:** Classification

## Problem statement

Identify low, medium, and high assessed burnout risk from pre-semester and study-behaviour information while avoiding post-outcome leakage.

This notebook is standalone: it contains its own loading, EDA, preprocessing, modelling, validation, interpretation, clustering, limitations, and recommendations. It writes no result files.

## Evidence boundary

The Kaggle source does not document collection or real-versus-synthetic provenance. Results are predictive associations within this file, not causal evidence about student outcomes.

## Aim and objectives

1. Build a leakage-controlled predictive baseline.
2. Compare at least three real models with a dummy baseline.
3. quantify cross-validation and test uncertainty.
4. inspect subgroup performance and predictor sensitivity.
5. add outcome-free student-profile clustering for unsupervised analysis.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
RUN_BALANCED_BACKUP = True
CV_FOLDS = 3 if RUN_BALANCED_BACKUP else 5
BOOTSTRAP_ITERATIONS = 200 if RUN_BALANCED_BACKUP else 1000

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)
print(f"Loaded {len(df):,} rows. CV folds: {CV_FOLDS}. Bootstrap iterations: {BOOTSTRAP_ITERATIONS}.")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

## Complete data audit and EDA

In [ ]:
audit = pd.Series({
    "rows": len(df),
    "columns_original": 16,
    "missing_cells": int(df.iloc[:, :16].isna().sum().sum()),
    "duplicate_rows": int(df.iloc[:, :16].duplicated().sum()),
    "unique_student_ids": int(df["Student_ID"].nunique()),
})
display(audit.to_frame("value"))
display(df.describe(include="all").T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="Burnout_Risk_Level", order=["Low", "Medium", "High"], ax=axes[0])
sns.histplot(data=df, x="Skill_Retention_Score", bins=30, kde=True, ax=axes[1])
sns.histplot(data=df, x="GPA_Change", bins=30, kde=True, ax=axes[2])
fig.tight_layout()
plt.show()

## Preprocessing contract

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(frame, scale_numeric=True):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical,
            ),
        ]
    )

In [ ]:
selected_features = EARLY_RISK_FEATURES
forbidden = {"Student_ID", "Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"}
assert not forbidden.intersection(selected_features)
print("Selected features:", selected_features)
print("Target distribution:")
display(df["Burnout_Risk_Level"].value_counts(normalize=True).rename("share").to_frame())

## Model comparison with cross-validation and untouched test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score, average_precision_score,
)

def classification_metrics(y_true, prediction, probability, classes):
    result = {
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "macro_f1": f1_score(y_true, prediction, average="macro"),
    }
    if len(classes) == 2:
        positive_index = list(classes).index(1)
        result["roc_auc"] = roc_auc_score(y_true, probability[:, positive_index])
        result["average_precision"] = average_precision_score(
            y_true, probability[:, positive_index]
        )
    else:
        result["macro_roc_auc_ovr"] = roc_auc_score(
            y_true, probability, multi_class="ovr",
            average="macro", labels=classes
        )
    return result

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

X = df[EARLY_RISK_FEATURES]
y = df["Burnout_Risk_Level"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(
        C=1.0, max_iter=1500, class_weight="balanced"
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=120 if RUN_BALANCED_BACKUP else 250,
        min_samples_leaf=3, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=100 if RUN_BALANCED_BACKUP else 180,
        learning_rate=0.08, max_leaf_nodes=31, random_state=RANDOM_STATE
    ),
}
rows = []
fitted_models = {}
for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor(X_train)),
        ("model", estimator),
    ])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"macro_f1": "f1_macro", "balanced_accuracy": "balanced_accuracy"},
        n_jobs=-1,
    )
    pipe.fit(X_train, y_train)
    prediction = pipe.predict(X_test)
    probability = pipe.predict_proba(X_test)
    row = {
        "model": name,
        "cv_macro_f1_mean": scores["test_macro_f1"].mean(),
        "cv_macro_f1_std": scores["test_macro_f1"].std(ddof=1),
        **classification_metrics(y_test, prediction, probability, pipe.classes_),
    }
    rows.append(row)
    fitted_models[name] = pipe
results = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
display(results)
best_name = results.iloc[0]["model"]
best_model = fitted_models[best_name]
best_prediction = best_model.predict(X_test)
best_probability = best_model.predict_proba(X_test)
print("Selected model:", best_name)
print(classification_report(y_test, best_prediction))

## Hyperparameter tuning

Tune one competitive model inside cross-validation, then evaluate the refitted configuration once on the untouched test set.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

tuning_pipeline = Pipeline([
    ("preprocess", make_preprocessor(X_train)),
    ("model", RandomForestClassifier(
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])
tuning_space = {
    "model__n_estimators": [100, 180, 260],
    "model__max_depth": [None, 8, 14],
    "model__min_samples_leaf": [2, 4, 8],
    "model__max_features": ["sqrt", 0.7],
}
tuning_search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions=tuning_space,
    n_iter=4 if RUN_BALANCED_BACKUP else 10,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    random_state=RANDOM_STATE,
    refit=True,
)
tuning_search.fit(X_train, y_train)
tuned_prediction = tuning_search.predict(X_test)
tuned_probability = tuning_search.predict_proba(X_test)
print("Best tuning parameters:", tuning_search.best_params_)
print("Best cross-validation macro-F1:", tuning_search.best_score_)
display(pd.DataFrame([{
    "model": "Tuned random forest",
    **classification_metrics(
        y_test, tuned_prediction, tuned_probability, tuning_search.classes_
    ),
}]))

## Bootstrap uncertainty

In [ ]:
from sklearn.utils import resample

rng = np.random.default_rng(RANDOM_STATE)
bootstrap_scores = []
positions = np.arange(len(y_test))
for _ in range(BOOTSTRAP_ITERATIONS):
    selected = rng.choice(positions, size=len(positions), replace=True)
    bootstrap_scores.append(
        f1_score(
            np.asarray(y_test)[selected],
            np.asarray(best_prediction)[selected],
            average="macro",
        )
    )
print("Bootstrap 95% CI for test macro-F1:",
      np.quantile(bootstrap_scores, [0.025, 0.975]))

## Feature-policy ablation and leakage audit

In [ ]:
ablation_rows = []
for label, features in {
    "Primary early-risk set": EARLY_RISK_FEATURES,
    "Expanded set with exam anxiety": EXPANDED_FEATURES,
}.items():
    ablation_model = Pipeline([
        ("preprocess", make_preprocessor(df[features])),
        ("model", LogisticRegression(max_iter=1500, class_weight="balanced")),
    ])
    X_ab_train, X_ab_test, y_ab_train, y_ab_test = train_test_split(
        df[features], df["Burnout_Risk_Level"], test_size=0.20,
        random_state=RANDOM_STATE, stratify=df["Burnout_Risk_Level"]
    )
    ablation_model.fit(X_ab_train, y_ab_train)
    ablation_rows.append({
        "feature_policy": label,
        "macro_f1": f1_score(
            y_ab_test, ablation_model.predict(X_ab_test), average="macro"
        ),
    })
display(pd.DataFrame(ablation_rows))

## Subgroup validation

In [ ]:
test_with_predictions = X_test.copy()
test_with_predictions["actual"] = np.asarray(y_test)
test_with_predictions["predicted"] = best_prediction
subgroup_rows = []
for column in ["Major_Category", "Year_of_Study", "Institutional_Policy"]:
    for value, group in test_with_predictions.groupby(column, observed=True):
        subgroup_rows.append({
            "subgroup_field": column,
            "subgroup": value,
            "rows": len(group),
            "macro_f1": f1_score(group["actual"], group["predicted"], average="macro"),
            "balanced_accuracy": balanced_accuracy_score(group["actual"], group["predicted"]),
        })
display(pd.DataFrame(subgroup_rows))

## Permutation importance

In [ ]:
from sklearn.inspection import permutation_importance

importance_sample = X_test.sample(n=min(3000, len(X_test)), random_state=RANDOM_STATE)
importance_target = y_test.loc[importance_sample.index]
importance = permutation_importance(
    best_model, importance_sample, importance_target,
    scoring="f1_macro", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
display(pd.DataFrame({
    "feature": importance_sample.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False))

## Predictor-only unsupervised student profiles

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

cluster_features = [
    "Pre_Semester_GPA", "Weekly_GenAI_Hours", "Tool_Diversity",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Anxiety_Level_During_Exams",
]
cluster_sample = df.sample(
    n=min(10000 if RUN_BALANCED_BACKUP else 20000, len(df)),
    random_state=RANDOM_STATE,
).copy()
cluster_scaled = StandardScaler().fit_transform(cluster_sample[cluster_features])
cluster_scores = []
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = candidate.fit_predict(cluster_scaled)
    cluster_scores.append({
        "k": k,
        "silhouette": silhouette_score(
            cluster_scaled, labels, sample_size=min(4000, len(cluster_sample)),
            random_state=RANDOM_STATE,
        ),
    })
cluster_scores = pd.DataFrame(cluster_scores)
selected_k = int(cluster_scores.loc[cluster_scores["silhouette"].idxmax(), "k"])
cluster_model = KMeans(n_clusters=selected_k, n_init=20, random_state=RANDOM_STATE)
cluster_sample["Cluster"] = cluster_model.fit_predict(cluster_scaled)
display(cluster_scores)
display(cluster_sample.groupby("Cluster", observed=True)[
    cluster_features + ["GPA_Change", "Skill_Retention_Score"]
].mean().round(2))
display(pd.crosstab(
    cluster_sample["Cluster"], cluster_sample["Burnout_Risk_Level"], normalize="index"
).round(3))

## Optional GPU extension

In [ ]:
# Optional GPU experiment. It is intentionally opt-in so the notebook remains CPU-safe.
RUN_GPU_NEURAL_MODEL = False
if RUN_GPU_NEURAL_MODEL:
    try:
        import torch
        print("PyTorch:", torch.__version__)
        print("CUDA available:", torch.cuda.is_available())
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Selected device:", device)
        print("Use the already transformed training arrays to define and train a task-specific PyTorch network.")
    except ImportError:
        print("PyTorch is not installed. Use 00_gpu_runtime_diagnostics.ipynb for setup guidance.")
else:
    print("GPU neural model skipped. Set RUN_GPU_NEURAL_MODEL=True after checking 00_gpu_runtime_diagnostics.ipynb.")

## Critical analysis, recommendations, and limitations

Use macro-F1 and class-level recall as the primary evidence. Recommendations should identify student profiles for further support or investigation, not claim that AI behaviour caused the outcome.

- Perfect cleanliness reduces the opportunity to demonstrate missing-data repair.
- Provenance, sampling, geography, and real-versus-synthetic status are undocumented.
- Cross-sectional associations do not establish temporal order or causality.
- Subgroup gaps are diagnostic and require confirmation with independently collected data.
- The final report should compare model gains against the dummy baseline, confidence intervals, and class-specific errors rather than accuracy alone.

## Conclusion

The preferred model is selected from consistent cross-validation and untouched-test evidence. The notebook demonstrates supervised and unsupervised learning while keeping the evidence boundary explicit.